In [ ]:
# 1. GPU (T4等) が利用可能か自動チェック
import sys
import torch

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPUが有効です: {gpu_name}")
else:
    # CPUの場合は警告を出す
    print("⚠️ 警告: GPUが有効になっていません！")
    print("Colabの上部メニュー [ランタイム] -> [ランタイムのタイプを変更] から『T4 GPU』を選択してください。")

AI演習：履歴を残す画像生成エージェント
Qwen3.5-4B（Ollama）が日本語の命令を英語プロンプトに書き換え、Stable Diffusionで描きます。
履歴を残すので、「もっと明るく」「猫を2匹に」のような修正が前回の続きにする

In [ ]:
# Ollamaをインストール（zstd・pciutilsはインストールに必要な補助ツール）

!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# Ollamaサーバーをバックグラウンドで起動
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # 起動を待つ

# -------------------------------------------------------
# Qwen3.5-4Bをダウンロード（8bit版。4bit量子化は使わない）
# -------------------------------------------------------
MODEL = "qwen3.5:4b-q8_0"
!ollama pull {MODEL}

Stable Diffusionの準備

In [ ]:
!pip install -q ollama diffusers

import torch
from diffusers import AutoPipelineForText2Image

# SD-Turbo
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")

T4 GPUの使用量を調べる

システムプロンプトと履歴

In [ ]:
import ollama

# AIの役割を決める
SYSTEM_PROMPT = """
あなたはStable Diffusion用のプロンプトを書く係です。
ユーザーの日本語の要望を、英語のプロンプトに書き換えてください。
・出力は英語のプロンプト1行だけ。説明や前置きは書かない。
・30語以内で、短く書く。
・修正の指示（例：「もっと明るく」）が来たら、前回のプロンプトを元に、その部分だけ直す。
"""

# ★ 履歴：会話がすべてここに溜まっていく
history = [{"role": "system", "content": SYSTEM_PROMPT}]

In [ ]:
# 要望 → 英語プロンプト（履歴を使う）
def make_prompt(request):
    history.append({"role": "user", "content": request})       # ① 要望を履歴に足す
    res = ollama.chat(model=MODEL, messages=history, think=False,
                      keep_alive=-1)                             # ② 履歴ごと送る（載せたまま）
    prompt = res.message.content.strip()
    history.append({"role": "assistant", "content": prompt})   # ③ 答えも履歴に足す
    return prompt

# 英語プロンプト → 画像
def draw(prompt, seed):
    g = torch.Generator("cuda").manual_seed(seed)  # 同じseedなら構図が変わりにくい
    return pipe(prompt, num_inference_steps=4, guidance_scale=0.0, generator=g).images[0]

描画の実行　（request を書き換えて実行）

描かせる（このセルを何度も実行する）

最初から構図ごと変えたいときは SEED の数字を変える。


In [ ]:
request = "夕焼けの海岸を歩く猫"
SEED = 42

prompt = make_prompt(request)
print("英語プロンプト:", prompt)
draw(prompt, SEED)

In [ ]:
# T4GPUの使用量を書き出す

import subprocess

# nvidia-smi に「使用量・全体量」だけを数値で出させる
out = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True
).stdout.strip()

used, total = [int(x) for x in out.split(",")]   # 単位は MiB
print(f"GPU使用量：{used/1024:.1f} GB / {total/1024:.1f} GB（{used/total*100:.0f}%）")

GPU使用量：9.1 GB / 15.0 GB（61%）


5. 履歴を見る/消す

In [ ]:
# 今、AIに渡している履歴をすべて表示
for m in history:
    print(f"[{m['role']}] {m['content']}\n")

[system] 
あなたはStable Diffusion用のプロンプトを書く係です。
ユーザーの日本語の要望を、英語のプロンプトに書き換えてください。
・出力は英語のプロンプト1行だけ。説明や前置きは書かない。
・30語以内で、短く書く。
・修正の指示（例：「もっと明るく」）が来たら、前回のプロンプトを元に、その部分だけ直す。


[user] 夕焼けの海辺を歩く猫

[assistant] Cat walking on seaside shore at sunset



In [ ]:
# 履歴を消して、最初からやり直す（systemだけ残す）
history = [{"role": "system", "content": SYSTEM_PROMPT}]
print("履歴をリセットしました")

履歴をリセットしました


6. 較べてみる：履歴を残さないとどうなるか

同じ修正の指示（例：もっと明るく）を、履歴なしで送ってみる。

In [ ]:
# 履歴なし版：毎回 system と今回の要望だけを送る
def make_prompt_no_history(request):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": request}]
    res = ollama.chat(model=MODEL, messages=messages, think=False)
    return res.message.content.strip()

prompt = make_prompt_no_history("もっと明るく")
print("英語プロンプト:", prompt)   # 何を明るくするのか、AIには分からない
draw(prompt, 42)